## What will you learn?

O que são modelos score driven para séries temporais. Como usar o pacote ScoerDrivenModels.jl de Julia para estimar e simular modelos GAS(p, q).

In this short article, you will learn the basics of a super flexible and general time series model framework called Score Driven Models. They were developed simultaneously by Harvey [2013] and Creal et al. [2013]. Known by different names - generalized autoregressive score (GAS), dynamic conditional score models, or simply score-driven models, which is the term chosen in this article - they offer several advantages for time series modeling, as described below. Firstly, these models can handle non-Gaussian time series, where the conditional probability distribution is not normal. This capability is crucial for cases where the normal distribution is inadequate, such as wind speed series (which only take positive values) and financial return series (which exhibit excess kurtosis). Count data, such as the number of items sold in a store, also benefit from this flexibility, as they require discrete distributions. The ability to model non-normal data makes GAS models highly applicable across various domains.

In this article, we will use a Natural Inflow Energy time series from Brazil, which is naturally non-gaussian, to exemplify the use of this type of model with an open-source package written in Julia called [ScoreDrivenModels.jl](https://lampspuc.github.io/ScoreDrivenModels.jl/latest//).

In order to keep this article short, I will not dive into the full lenght of the theory behind score driven models. To do so, I strongly recommend diving into the following resources:

 - [Gas Model Website](https://www.gasmodel.com/index.htm) with a lot of publised papers and material regarding score-driven models;
 - [Harvey original paper](https://www.cambridge.org/br/universitypress/subjects*/economics/econometrics-statistics-and-mathematical-economics/dynamic-models-volatility-and-heavy-tails-applications-financial-and-economic-time-series?format=PB&isbn=9781107630024) with the original description of the Dynamic Conditional Score model;
 - [Cral et al. original paper](https://doi.org/10.1002/jae.1279) with the original implementation of the GAS(p, q) model.

## What is a Score Driven Model?

The best way to introduce a score-driven model is to define the fundamental steps that we need to follow to model a time series with this framework. They are a probabilistic model, to shat out first step is to define a conditional probability distriution to be followed by the time series. Note that this step step is super general! We can choose any distribution that we think will best describe the nature of the time series.

![Conditional Probability Distribution](output/prob_definition.png)

This definition is saying the out time series $y_t$ follows a conditional probability distribution of both the past observations of the series, static parameters and time-varying parameters (which are of crucial importance to this framework). One this distribution is defined, we need to choose how the time-varying parameters will evolve over time, since they are (who would guess?) time-varying. There are two main options: (a) an ARMA(p, q) process, following Creal et al., giving origin to a GAS(p,q) model; or (b) an unobserved components framework, following Harvey's proposition. We will choose the first one, since it is the one implemented inside the [ScoreDrivenModels.jl] package.

The GAS(p,q) model says that the time-varying parameters follow an ARMA(p,q) process. Yes! the same from the ARIMA family! Actually, the GAS(p,q) model generalized both ARMA and GARCH models. Since we are imposing an ARMA(p, q) process in the time-varying parameters, this makes the GAS(p,q) framwework innapropriate for non-stationary time series. This is a core point of this methodology! The unobs erved components framework, on the other hand, is appropriate for non-stationary time series! 

![Gas(p,q)](output/gas_pq.png)

Wait a minute! There is a new symbol in this definition which we have not presented yet: $s_t$. This is, indeed, the core of the proposed framework. It is the $standard score$ of the time series. Let's define, first, who is the score: it is the first derivative of the log-likelihood function with respect to the parameter of interest. This has some intuitions: is says how much improvement we have on the likelihood of our model if we change the parameter of interest (mean, variance, and so on). It is the score that drives the dymanic of the score-driven model!

![Score](output/score_definition.png)

Finally, we can define the standardized score, which is simply a scaled version of the score obtained by a (highly non trivial) multiplication. I will not dive into the properties of this scaled version, but it is a key element of the framework.

![Std score](output/fisher_std_score_def.png)

The last piece of theory I would like to talk about is the model's estimation. As usual, it is done with maximun likelihood estimation, in which we aim to find the optimal parameters that maximize the likelihood of the model. This is particularly straightforward, provided an efficient nonlinear optimization algorithm is used, requiring only the implementation of the updating function and the evaluation of the likelihood function at a particular optimization point.

## The ScoreDrivenModels.jl package

[Link para a documentação](https://lampspuc.github.io/ScoreDrivenModels.jl/latest/).

[Link para o artigo](https://arxiv.org/abs/2008.05506).

This package, developed by our laboratory LAMPS implements score driven models of type GAS(p,q) for a variety of available distributions with 1 or more time-varying parameters.
Take a look into the package documentation to a full report of it's functionality! You can also dive into the package's article.
The table below summarizes the available distributions where the identity scale means d=0, inverse scale means d=-1 and inverse square means d=0.5.
Available distributionsMoreover, the package also allows for new distributions to be implemented.

![Distributions](output/package_distrib.png)

Moreover, the package also allows for new distributions to be implemented.
 There is a number of available optimizers to fit the model with MLE, such as Nelder Mead, LBFGS and IPNewton. However, there is a fundamental trick! 
Those optimizers are unconstrained algorithms, which means that they do not garantee any type of restrictions of the distribution parameters. This is a huge problem, since variance, for instance, can never be negative!
The trick to guarantee this type of restrictions is through link functions, which are described in the package documentation, here.

![Link](output/link_functions.png)


## Let's see it in action

First of all, we need the following packages:



In [1]:
import Pkg
path = pwd()*"/TimeSeriesJulia/ScoreDrivenModels/"
Pkg.activate(path)
Pkg.instantiate()

  Activating project at `c:\Users\matheuscn.ELE.000\Documents\Diversos\MathNogMediumArticles\TimeSeriesJulia\ScoreDrivenModels\TimeSeriesJulia\ScoreDrivenModels`
    Updating registry at `C:\Users\matheuscn.ELE.000\.julia\registries\General.toml`
   Installed ADTypes ────────────────── v1.18.0
   Installed ArrayInterface ─────────── v7.20.0
   Installed Adapt ──────────────────── v4.4.0
   Installed Compat ─────────────────── v4.18.1
   Installed Roots ──────────────────── v2.2.10
   Installed Rmath_jll ──────────────── v0.4.3+0
   Installed FiniteDiff ─────────────── v2.28.1
   Installed HypothesisTests ────────── v0.10.13
   Installed SpecialFunctions ───────── v2.6.1
   Installed ScoreDrivenModels ──────── v0.2.1
   Installed Distributions ──────────── v0.25.122
   Installed DifferentiationInterface ─ v0.7.8
    Updating `C:\Users\matheuscn.ELE.000\Documents\Diversos\MathNogMediumArticles\TimeSeriesJulia\ScoreDrivenModels\TimeSeriesJulia\ScoreDrivenModels\Project.toml`
  [31c24e10] 

In [2]:
# Pkg.add("Dates")
# Pkg.add("Plots")
# Pkg.add("DelimitedFiles")
# Pkg.add("Distributions")
# Pkg.add("ScoreDrivenModels")

using Dates
using Plots
using DelimitedFiles
using Distributions
using ScoreDrivenModels

# plotlyjs()

### Loading a time series

This is a monthly time series of the natural inflow energy of the Northeast region of Brazil, available inside the package.

I have dowloaded it from the package repo and saved it in the `data` folder in my [GitHub repository](https://github.com/MathNog/MathNogMediumArticles).

In [3]:
dates = collect(Date(1961):Month(1):Date(2000, 12));
y = vec(readdlm("data/nie_northeastern.csv"));

H  = 60
T  = 240
last_obs  = length(y) - H
first_obs = last_obs - T

y_train     = y[first_obs:last_obs];
y_test      = y[last_obs+1:end];

dates_train = dates[first_obs:last_obs];
dates_test  = dates[last_obs+1:end];

In [4]:
plot(dates_train, y_train, label = "Train")
plot!(dates_test, y_test, label = "Test") 
plot!(title="Brazil Northeastern Natural Inflow Energy", xlabel="Date", ylabel="Value")
plot!(xformatter = x -> Dates.format(Date(Dates.UTD(x)), "yyyy"))
savefig("output/northeastern_ts.png");

### Defining the model

We define the model using the `ScoreDrivenModel` function from the `ScoreDrivenModels` package. The function takes the following arguments:

- `p_lags`: The lags of the autoregressive part of the model.
- `q_lags`: The lags of the moving average part of the model.
- `dist`: The distribution of the model.
- `d`: The degree of the model.
- `time_varying_params`: The parameters of the model that are time-varying.


This is a crucial part of the modeling process, as it is when, based on the time-series features, we define the model's structure and the parameters that we want to estimate.

Since our time series is of monthly frequency and it is of a time series of natural inflow energy, we expect to have a strong seasonal pattern (as shown in the previous plot). Because of that, we will add a seasonal lag of 12 months in both the autoregressive and moving average parts of the model.

Moreover, since natural inflow can never be negative, it is natural to use a distribution that is always positive, such as the log-normal distribution. It is important to say that there is no technical reason not to use a Normal distribution. The model would be fitted and simulated without any issues, but we would possibly generate negative scenarios, which would not make sense in this context!

Finally, we will fit a Log-normal GAS model with both parameters as time varying, that is, both will follow an ARMA process with autoregressive and moving average lags of 1 and 12 months. 

In [5]:
p_lags  = [1, 12]
q_lags  = [1, 12]
distrib = Distributions.LogNormal
d       = 0.0

gas = ScoreDrivenModel(p_lags, q_lags, distrib, d; time_varying_params = [1, 2]);

### Estimating and Simulating the model

We can fit and forecast the model with two simple functions! Notice that we are doing a probabilistic forecast with 1000 scenarios!

In [6]:
fit_gas = fit!(gas, y_train);
forecast_gas = forecast(y_train, gas, H; S=1000);

┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matheuscn.ELE.000\.julia\packages\Optim\7krni\src\types.jl:120


Round 1 of 3: log-likelihood = -531.8207714482409
Round 2 of 3: log-likelihood = -510.2838718585067


┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matheuscn.ELE.000\.julia\packages\Optim\7krni\src\types.jl:120
┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matheuscn.ELE.000\.julia\packages\Optim\7krni\src\types.jl:120


Round 3 of 3: log-likelihood = -539.5306715170992


### Visualize the fitted values and the simulated scenarios

Once the model is fitted and forecasted, we can visualize it's results!

First, the fitted parameters.

In [7]:
results_gas = results(fit_gas)

--------------------------------------------------------
Distribution:                 LogNormal
Number of observations:       241
Number of unknown parameters: 10
Log-likelihood:               -510.2839
AIC:                          1040.5677
BIC:                          1075.4157


--------------------------------------------------------
Parameter      Estimate   Std.Error     t stat   p-value
omega_1          0.0783      0.6688     0.1171    0.9091
omega_2         -0.0490      0.0000 -4903.1146    0.0000
A_1_11           0.0267      0.0747     0.3577    0.7280
A_1_22           0.0018      0.2811     0.0065    0.9949
A_12_11          0.0684      0.1255     0.5451    0.5977
A_12_22          0.0711      0.2603     0.2730    0.7904
B_1_11           0.6140      1.0428     0.5888    0.5691
B_1_22           0.9870      0.0000 98697.8485    0.0000
B_12_11          0.3324      0.9726     0.3417    0.7396
B_12_22         -0.0060      0.0000  -597.4948    0.0000


We can also plot the fitted mean and both point and probabilistic forecasts.

In [8]:
point_forecast = forecast_gas.observation_forecast;
scenarios      = forecast_gas.observation_scenarios;
quantiles_obs  = forecast_gas.observation_quantiles;
fitted_values  = fitted_mean(gas, y_train);

In [9]:
plot(dates_train, y_train, label = "Train Values", color = :black)
plot!(dates_train, fitted_values, label = "Fitted Values", color = :blue)
plot!(dates_test, scenarios, label = "", color = :lightgray)
plot!(dates_test, y_test, label = "Test Values", color = :black)
plot!(dates_test, point_forecast, label = "Point Forecast", color = :red)
title!("Natural Inflow Energy Simulation from LogNormal GAS")
plot!(xformatter = x -> Dates.format(Date(Dates.UTD(x)), "yyyy"),
    legendcolumns = 4, legend = :outerbottom)
savefig("output/gas_simulation.png")

"c:\\Users\\matheuscn.ELE.000\\Documents\\Diversos\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\gas_simulation.png"

### Visualize the fitted parameters and the simulated scenarios for each parameter

Even thought this may seen enough to present a time series modeling and forecasting package, there are some super interesting and powerful functionalities implemented that must be talked about!

First, we can visualize each parameter fitted values and probabilistic forecast! Yes, even thought variance is, essentially, unobserved, we can visualize it with this package as follows.

In [10]:
params_fitted = score_driven_recursion(gas, y_train)
params_point_forecast = forecast_gas.parameter_forecast; #H x P
params_scenarios      = forecast_gas.parameter_scenarios; #H x P x S

In [11]:
for i in 1:size(params_fitted, 2)
	plot(dates_train, params_fitted[2:end, i], label = "Fitted Parameters", color = :blue)
	plot!(dates_test, params_scenarios[:, i, :], label = "", color = :lightgray)
	plot!(dates_test, params_point_forecast[:, i], label = "Point Forecast", color = :red)
    title!("Parameter $i for LogNormal GAS")
    savefig("output/parameter_$i.png")
end

### Visualize the residuals of the model: Pearson and Quantile Residuals

In [12]:
q_residuals   = quantile_residuals(y_train, gas);
std_residuals = pearson_residuals(y_train, gas);

In [28]:
plot_quantile_residuals = plot(dates_train[25:end], q_residuals, title = "Quantile Residuals", xlabel = "Date", ylabel = "Value", label = "")
savefig(plot_quantile_residuals, "output/q_residuals.png")
plot_pearson_residuals = plot(dates_train[25:end], std_residuals, title = "Pearson Residuals", xlabel = "Date", ylabel = "Value", label = "")
savefig(plot_pearson_residuals, "output/pearson_residuals.png")
plot(plot_quantile_residuals, plot_pearson_residuals, layout = (2,1), size = (1000, 600),
    title = ["Quantile Residuals" "Pearson Residuals"], xlabel = "Date", ylabel = "", label = "",
    xformatter = x -> Dates.format(Date(Dates.UTD(x)), "yyyy"))
savefig("output/residuals.png")


"c:\\Users\\matheuscn.ELE.000\\Documents\\Diversos\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\residuals.png"

### Built-in Time Series Cross Validation

There is a built-in function for time series cross validation, in which we simply define the forecast horizon and the first timestamp index to fit the model. This function automatically computed the number of windows to fit and forecast the model, while returning different accuracy metrics across all windows and horizons.

In [14]:
first_index = 100
gas_cv = cross_validation(gas, y_train, H, first_index);

CrossValidation: step 1 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 2 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 3 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 4 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 5 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 6 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 7 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 8 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 9 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 10 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 11 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 12 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 13 of 81
Score Driven Model does not have unknowns.
CrossValidation: step 14 of 81
Score Driven Mod

In [15]:
cv_abs_errors = gas_cv.abs_errors
cv_mean_crps  = gas_cv.mean_crps
cv_mae        =  gas_cv.mae

H, W = size(cv_abs_errors)

(60, 81)

In [16]:
plot_abs = plot(1:H, mean(cv_abs_errors, dims=2), title = "Mean Absolute Error across all $W windows by $H steps ahead", xlabel = "Window", ylabel = "Mean Absolute Error", label = "")
savefig("output/abs_errors.png")
plot_crps = plot(1:H, cv_mean_crps, dims=2, title = "Mean CRPS across all $W windows by $H steps ahead", xlabel = "Window", ylabel = "Mean CRPS", label = "")
savefig("output/crps.png")
plot_mae = plot(1:H, cv_mae, title = "MAE across all $W windows by $H steps ahead", xlabel = "Window", ylabel = "MAE", label = "")
savefig("output/mae.png")

"c:\\Users\\matheuscn.ELE.000\\Documents\\Diversos\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\mae.png"

In [21]:
plot(1:H, cv_mean_crps, dims=2, label="Mean CRPS")
plot!(1:H, cv_mae, label="MAE")
plot!(xlabel = "Window", ylabel = "Error", title = "Mean CRPS and MAE across all $W windows by $H steps ahead",
titlefontsize = 11)
savefig("output/all_errors.png")

"c:\\Users\\matheuscn.ELE.000\\Documents\\Diversos\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\all_errors.png"

### Residuals Diagnostics

In [20]:
plot(fit_gas, size=(1000, 1000), title = ["Residuals" "ACF Plot" "Histogram" "QQ Plot"])
savefig("output/residuals_diagnostics.png")

"c:\\Users\\matheuscn.ELE.000\\Documents\\Diversos\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\residuals_diagnostics.png"